# Encoder-Only Transformer (From Scratch)

This notebook implements a lightweight transformer encoder, built entirely from scratch (no pretrained weights), to predict the correct answer among 5 options for each MCQ prompt.

**Approach:** each (prompt, option) pair is tokenized using a sinple, custom word-level vocabulary, embedded, and passed through a Transformer encoder with sinusoidal positional encoding. The 5 resulting option representations are pooled and scored, and the model is trained to rank the correct option highest via cross-entropy loss.

# Importing Libraries

In [1]:
import os
import re
import random
import math
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", palette="muted")

Using device: cuda


# Loading Dataset

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

Datasets loaded!


# Define the Model

## Configuration

In [3]:
@dataclass
class Config:
    model_name: str = "transformer-scratch"
    vocab_size: int = 4000 
    embed_dim: int = 128
    hidden_dim: int = 256
    max_seq_len: int = 128 
    epochs: int = 30
    batch_size: int = 32  
    
    learning_rate: float = 1e-3 
    weight_decay: float = 0.01

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    wandb_run_name: str = "transformer-scratch"
    
    def to_dict(self):
        return asdict(self)

cfg = Config()

In [4]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adrija935 (23f2004791-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into Weights & Biases!


## Vocabulary Builder

In [5]:
class SimpleVocab:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1} 
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq
        self.vocab_size = 2

    def tokenize(self, text):
        # Lowercase and remove punctuation
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()

    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            word_counts.update(self.tokenize(text))
            
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text, max_len=128):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(word, 1) for word in tokens] 
        if len(ids) > max_len:
            ids = ids[:max_len] # Clipping
        else:
            ids = ids + [0] * (max_len - len(ids)) # Padding
        return ids


# Collect all text to build vocabulary
corpus = train_df['prompt'].tolist()
for col in ['A', 'B', 'C', 'D', 'E']:
    corpus.extend(train_df[col].tolist())
    
vocab = SimpleVocab(min_freq=2)
vocab.build_vocab(corpus)
cfg.vocab_size = vocab.vocab_size
print(f"Vocabulary size built: {vocab.vocab_size} unique tokens.")

Vocabulary size built: 3081 unique tokens.


## PyTorch Dataset and DataLoaders

In [6]:
class MCQDataset(Dataset): 
    def __init__(self, df, vocab, max_len=128, is_test=False):
        self.df = df
        self.vocab = vocab 
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        options = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
        
        input_ids = []
        for opt in options:
            # Combine prompt and option text 
            text = prompt + " " + opt
            ids = self.vocab.encode(text, max_len=self.max_len)
            input_ids.append(ids)
            
        input_tensor = torch.tensor(input_ids, dtype=torch.long) 
        if not self.is_test:
            label = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            return input_tensor, label
        return input_tensor

train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=SEED, shuffle=True, stratify=train_df["answer"])

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print(f"Train samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

# Instantiate Datasets and DataLoaders
train_dataset = MCQDataset(train_split, vocab, max_len=cfg.max_seq_len)
val_dataset = MCQDataset(val_split, vocab, max_len=cfg.max_seq_len)
test_dataset = MCQDataset(test_df, vocab, max_len=cfg.max_seq_len, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

Train samples: 1600
Validation samples: 400


## Model Architecture

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.3, max_len=256):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class LightweightMCQTransformer(nn.Module):
    def __init__(
        self, 
        vocab_size, 
        embed_dim=128, 
        num_heads=4, 
        hidden_dim=256, 
        num_layers=2, 
        dropout=0.4, 
        max_len=128
    ):
        super(LightweightMCQTransformer, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim, dropout, max_len)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=hidden_dim, 
            dropout=dropout, 
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.norm = nn.LayerNorm(embed_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        batch_size, num_options, seq_len = x.shape
        
        x = x.view(batch_size * num_options, seq_len)
        
        src_key_padding_mask = (x == 0) 
        
        embedded = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)

        embedded = self.pos_encoder(embedded.transpose(0, 1)).transpose(0, 1)
        
        # Pass through Transformer
        transformer_out = self.transformer_encoder(
            embedded, 
            src_key_padding_mask=src_key_padding_mask
        )
        
        # Mean Pooling over unmasked tokens
        mask = (~src_key_padding_mask).unsqueeze(-1).float()
        sum_embeddings = torch.sum(transformer_out * mask, dim=1)
        valid_lengths = torch.clamp(mask.sum(1), min=1e-9)
        pooled_out = sum_embeddings / valid_lengths
        
        pooled_out = self.norm(pooled_out)
        
        # Score each option
        scores = self.fc(pooled_out)
        
        logits = scores.view(batch_size, num_options)
        
        return logits

## Scoring Metric (MAP@3)

In [8]:
# MAP@3 Metric
def compute_map_at_3(predictions, targets):
    scores = []
    for top_preds, target in zip(predictions, targets):
        score = 0.0
        for rank, pred in enumerate(top_preds):
            if pred == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Training and Validation

In [9]:
wandb.init(project=cfg.wandb_project, name=cfg.wandb_run_name, config=cfg.to_dict(), reinit=True)

model = LightweightMCQTransformer(vocab_size=vocab.vocab_size, embed_dim=cfg.embed_dim, hidden_dim=cfg.hidden_dim, num_heads=4, num_layers=2).to(cfg.device)

criterion = nn.CrossEntropyLoss()

param_optimizer = list(model.named_parameters())
no_decay = ['bias', 'norm']

optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
     'weight_decay': cfg.weight_decay},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
     'weight_decay': 0.01}
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=1e-3, betas=(0.9, 0.98))

total_steps = len(train_loader) * cfg.epochs
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=2e-3,
    total_steps=total_steps,
    pct_start=0.05, # Warm up for the first 5% of training
    anneal_strategy='cos'
)

best_map3 = 0.0

for epoch in range(cfg.epochs):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
    for inputs, labels in loop:
        inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping prevents gradient saturation/explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    train_loss = running_loss / len(train_loader.dataset)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    all_top3_preds = []
    all_top1_preds = []
    all_targets = []
    
    with torch.no_grad():
        val_loop = tqdm(val_loader)
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)

            top1_preds = torch.argmax(outputs, dim=1)
            _, top3_indices = torch.topk(outputs, k=3, dim=1)

            all_top1_preds.append(top1_preds.cpu().numpy())
            all_top3_preds.append(top3_indices.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    all_top1_preds = np.concatenate(all_top1_preds, axis=0)
    all_top3_preds = np.concatenate(all_top3_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    val_map3 = compute_map_at_3(all_top3_preds, all_targets)
    val_accuracy = accuracy_score(all_targets, all_top1_preds)
    val_f1 = f1_score(all_targets, all_top1_preds, average='macro')

    print(f"Epoch {epoch+1}/{cfg.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val MAP@3: {val_map3:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map3": val_map3,
        "val_accuracy": val_accuracy,
        "val_f1": val_f1
    })

    if val_map3 > best_map3:
        best_map3 = val_map3

wandb.run.summary["best_map3"] = best_map3
wandb.run.summary["best_accuracy"] = val_accuracy
wandb.run.summary["best_f1"] = val_f1
        
torch.save(model.state_dict(), "epoch_30_transformer_model.pt")

wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run e4lkdoyb
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260803_050941-e4lkdoyb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run transformer-scratch
wandb: ⭐️ View project at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: 🚀 View run at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/e4lkdoyb
  0%|          | 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src

Epoch 1/30 | Train Loss: 1.6209 | Val Loss: 1.1933 | Val MAP@3: 0.8337 | Val Acc: 0.7225 | Val F1: 0.7182


100%|██████████| 13/13 [00:00<00:00, 93.19it/s]


Epoch 2/30 | Train Loss: 0.7571 | Val Loss: 0.1392 | Val MAP@3: 0.9838 | Val Acc: 0.9725 | Val F1: 0.9713


100%|██████████| 13/13 [00:00<00:00, 91.13it/s]


...

100%|██████████| 13/13 [00:00<00:00, 91.98it/s]


Epoch 29/30 | Train Loss: 0.0229 | Val Loss: 0.0150 | Val MAP@3: 0.9938 | Val Acc: 0.9875 | Val F1: 0.9879


100%|██████████| 13/13 [00:00<00:00, 92.62it/s]
wandb: updating run metadata


Epoch 30/30 | Train Loss: 0.0145 | Val Loss: 0.0150 | Val MAP@3: 0.9938 | Val Acc: 0.9875 | Val F1: 0.9879


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:        epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:   train_loss █▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: val_accuracy ▁█████████████████████████████
wandb:       val_f1 ▁█████████████████████████████
wandb:     val_loss █▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     val_map3 ▁█████████████████████████████
wandb: 
wandb: Run summary:
wandb: best_accuracy 0.9875
wandb:       best_f1 0.98791
wandb:     best_map3 0.995
wandb:         epoch 30
wandb:    train_loss 0.01445
wandb:  val_accuracy 0.9875
wandb:        val_f1 0.98791
wandb:      val_loss 0.01499
wandb:      val_map3 0.99375
wandb: 
wandb: 🚀 View run transformer-scratch at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/e4lkdoyb
wandb: ⭐️ View project at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wan

# Inference

In [10]:
model.eval()

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
submission_preds = []

with torch.no_grad():
    for inputs in test_loader:
        inputs = inputs.to(cfg.device)
        outputs = model(inputs)
        _, top3_indices = torch.topk(outputs, k=3, dim=1)
        
        for top_three in top3_indices.cpu().numpy():
            pred_str = " ".join([idx_to_label[i] for i in top_three])
            submission_preds.append(pred_str)

# Generate Submission

In [11]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'!")
print(submission_df.head())

Submission saved to 'submission.csv'!
   id Prediction
0   1      A E B
1   2      B C D
2   3      B E D
3   4      E C D
4   5      C A B
